# 03 — MAS_RAG: Multi-Agent System with Retrieval

```
User Question + Retrieved Evidence (top-k=5)
     │
     ▼
┌──────────────────────────────────────────┐
│ 1. DOMAIN_EXPERT                          │
│    Interprets business rules & terminology │
└──────────────────────────────────────────┘
     │
     ▼
┌──────────────────────────────────────────┐
│ 2. DATA_ENGINEER                          │
│    Resolves tables & join paths            │
└──────────────────────────────────────────┘
     │
     ▼
┌──────────────────────────────────────────┐
│ 3. SQL_DEVELOPER                          │
│    Writes & executes SQL (retry up to 2x)  │
└──────────────────────────────────────────┘
     │
     ▼
┌──────────────────────────────────────────┐
│ 4. QUANTITATIVE_ANALYST                   │
│    Interprets SQL results numerically      │
└──────────────────────────────────────────┘
     │
     ▼
┌──────────────────────────────────────────┐
│ 5. QUALITY_AUDITOR                        │
│    Validates groundedness, final answer    │
└──────────────────────────────────────────┘
     │
     ▼
  Final Answer
```

## Properties
- 5 specialized expert agents in a fixed sequential pipeline
- Same retrieval as SAS_RAG (top-5 chunks, threshold 0.3)
- Each agent sees the question + schema + retrieved evidence + all previous agents' outputs
- SQL_SAFETY_RULES enforced in SQL_DEVELOPER prompt (same rules as SAS/SAS_RAG)
- SQL extraction failure triggers retry (not raw response as SQL)
- SQL_DEVELOPER retries up to 2 times on failure
- CANNOT_ANSWER from any pre-QA agent terminates the pipeline early
- QUALITY_AUDITOR: only startswith("CANNOT_ANSWER") triggers abstention
- Technical errors produce answer=None (not error strings)
- Experiment loop: try/except per question (crash-safe)

In [0]:
import time
import json
import re
import uuid
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================
ARCHITECTURE = "MAS_RAG"

# LLM client — shared from mt_config (429-resilient, max_retries=5)
client = LLM_CLIENT

MODEL_ENDPOINT = MT_MODEL_ENDPOINT

# Build base context (technical schema only — semantic retrieved via RAG)
schema_context_dict = build_unified_context(ARCHITECTURE)
schema_context_str = format_unified_context_for_prompt(schema_context_dict, include_semantic=False)

# ============================================================
# WORKFLOW STATE — Tracks all 5 agent contributions
# ============================================================

@dataclass
class WorkflowState:
    """Shared state passed between all 5 expert agents."""
    question: str = ""
    question_id: str = ""
    # Retrieval
    retrieved_chunks: List[Dict] = field(default_factory=list)
    retrieved_context_str: str = ""
    # Agent contributions (accumulated text from each agent)
    agent_outputs: Dict[str, str] = field(default_factory=dict)
    # SQL execution
    sql_generated: Optional[str] = None
    sql_results: Optional[str] = None
    sql_execution_status: str = "not_attempted"
    sql_attempts: int = 0
    # Final output
    final_answer: Optional[str] = None
    can_answer: bool = True
    # Orchestration metadata
    steps_log: List[Dict] = field(default_factory=list)
    total_prompt_tokens: int = 0
    total_completion_tokens: int = 0
    llm_calls: int = 0
    status: str = "in_progress"
    error: Optional[str] = None


print(f"✓ MAS_RAG configured ({ARCHITECTURE})")
print(f"  Agents: {MAS_AGENT_ORDER}")
print(f"  Model:  {MODEL_ENDPOINT}")
print(f"  Schema context: {len(schema_context_str):,} chars")

In [0]:
# ============================================================
# CORE UTILITIES
# ============================================================

def execute_sql_query(sql: str) -> str:
    """Execute SQL and return results as formatted string."""
    try:
        result_df = spark.sql(sql).toPandas()
        n_rows = len(result_df)
        if n_rows > 30:
            return f"({n_rows} rows total, showing first 30)\n" + result_df.head(30).to_string(index=False)
        return result_df.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"


def extract_sql_from_response(text: str) -> str:
    """Extract SQL query from LLM response text."""
    match = re.search(r"```(?:sql)?\s*(.+?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    lines = text.strip().split("\n")
    sql_lines = []
    capture = False
    for line in lines:
        stripped = line.strip().upper()
        if stripped.startswith(("SELECT", "WITH")):
            capture = True
        if capture:
            sql_lines.append(line)
    if sql_lines:
        return "\n".join(sql_lines).strip()
    if any(kw in text.upper() for kw in ["SELECT", "FROM"]):
        return text.strip()
    return None


def _call_llm(system_prompt: str, user_prompt: str, params: dict) -> Tuple[str, int, int]:
    """Make one LLM call. Returns (response_text, prompt_tokens, completion_tokens)."""
    resp = client.chat.completions.create(
        model=MODEL_ENDPOINT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=params["temperature"],
        max_tokens=params["max_tokens"],
        **_seed_kwargs()
    )
    return (
        resp.choices[0].message.content,
        resp.usage.prompt_tokens,
        resp.usage.completion_tokens
    )


# ============================================================
# 5-EXPERT SEQUENTIAL PIPELINE
# ============================================================
# Each agent receives: question + schema + retrieved context + ALL previous agents' outputs
# Each agent has a genuine professional profile (from AGENT_PROFILES in mt_config)
# The SQL_DEVELOPER agent additionally executes SQL and retries on failure
# ============================================================

def _build_collaboration_context(state: WorkflowState) -> str:
    """Build accumulated context from all previous agent contributions."""
    if not state.agent_outputs:
        return ""
    parts = []
    for agent_name, output in state.agent_outputs.items():
        profile = AGENT_PROFILES[agent_name]
        parts.append(f"--- {agent_name} ({profile['description']}) ---\n{output}")
    return "\n\n".join(parts)


def run_agent(state: WorkflowState, agent_name: str, max_sql_retries: int = 2) -> WorkflowState:
    """
    Run a single expert agent in the pipeline.
    
    For SQL_DEVELOPER: generates SQL, executes it, retries on failure.
    For all others: provides analysis based on their expertise.
    """
    if not state.can_answer:
        return state

    profile = AGENT_PROFILES[agent_name]
    step_start = time.time()
    step_idx = len(state.steps_log) + 1

    # Build user prompt with full context + previous agents' work
    collab_context = _build_collaboration_context(state)
    
    base_context = (
        f"QUESTION: {state.question}\n\n"
        f"SCHEMA CONTEXT:\n{schema_context_str}\n\n"
        f"RETRIEVED EVIDENCE (RAG):\n{state.retrieved_context_str}\n"
    )
    
    if collab_context:
        base_context += f"\nPREVIOUS AGENTS' ANALYSIS:\n{collab_context}\n"
    
    # Add SQL results context for agents after SQL_DEVELOPER
    if state.sql_generated and agent_name in ["QUANTITATIVE_ANALYST", "QUALITY_AUDITOR"]:
        base_context += (
            f"\nSQL QUERY EXECUTED:\n{state.sql_generated}\n"
            f"SQL STATUS: {state.sql_execution_status}\n"
            f"SQL RESULTS:\n{state.sql_results or '(empty)'}\n"
        )

    # --- SQL_DEVELOPER: special handling with execution + retry ---
    if agent_name == "SQL_DEVELOPER":
        for attempt in range(1, max_sql_retries + 1):
            state.sql_attempts = attempt
            retry_context = ""
            if attempt > 1 and state.sql_execution_status == "failed":
                retry_context = (
                    f"\n\nPREVIOUS SQL ATTEMPT FAILED:\n"
                    f"SQL: {state.sql_generated}\n"
                    f"Error: {state.sql_results}\n"
                    f"Fix the query and try again."
                )
            
            _sql_rules = "\n".join(f"- {r}" for r in SQL_SAFETY_RULES)
            user_prompt = base_context + retry_context + f"\n\nSQL RULES:\n{_sql_rules}\n\nWrite the SQL query now:"
            
            response, pt, ct = _call_llm(profile["system_prompt"], user_prompt, profile["llm_params"])
            state.total_prompt_tokens += pt
            state.total_completion_tokens += ct
            state.llm_calls += 1
            
            # Check for NO_SQL_NEEDED
            if "NO_SQL_NEEDED" in response.upper() or "CANNOT_ANSWER" in response.upper():
                state.sql_execution_status = "not_applicable"
                state.agent_outputs[agent_name] = response
                break
            
            state.sql_generated = extract_sql_from_response(response)
            if not state.sql_generated:
                # Extraction failed — treat as SQL failure for retry
                state.sql_execution_status = "failed"
                state.sql_results = "SQL ERROR: Could not extract valid SQL from LLM response"
                continue  # retry with error feedback
            
            state.sql_results = execute_sql_query(state.sql_generated)
            
            if state.sql_results.startswith("SQL ERROR"):
                state.sql_execution_status = "failed"
            else:
                state.sql_execution_status = "success"
                state.agent_outputs[agent_name] = f"SQL: {state.sql_generated}\nResult: {state.sql_results}"
                break
        else:
            # All retries exhausted
            state.agent_outputs[agent_name] = f"SQL FAILED after {max_sql_retries} attempts: {state.sql_results}"
    
    # --- All other agents: standard LLM call ---
    else:
        user_prompt = base_context + f"\nProvide your expert analysis as {profile['description']}:"
        
        response, pt, ct = _call_llm(profile["system_prompt"], user_prompt, profile["llm_params"])
        state.total_prompt_tokens += pt
        state.total_completion_tokens += ct
        state.llm_calls += 1
        
        state.agent_outputs[agent_name] = response
        
        # Check for early termination signals (non-QA agents only).
        # QUALITY_AUDITOR has its own startswith logic below — the generic
        # check must NOT fire for QA, or it corrupts sql_execution_status.
        if "CANNOT_ANSWER" in response.upper() and agent_name != "QUALITY_AUDITOR":
            state.can_answer = False
            state.status = "abstained"
            state.final_answer = response
            state.sql_execution_status = "not_applicable"
        
        # QUALITY_AUDITOR is the final agent — its output IS the answer
        if agent_name == "QUALITY_AUDITOR":
            state.final_answer = response
            # Only abstain when the response STARTS with CANNOT_ANSWER
            # (meaning the auditor determined the schema cannot support the question).
            # INSUFFICIENT_EVIDENCE and HALLUCINATION_DETECTED should NOT trigger
            # abstention — the auditor should still produce a best-effort answer.
            response_trimmed = response.strip().upper()
            if response_trimmed.startswith("CANNOT_ANSWER"):
                state.status = "abstained"
            else:
                state.status = "success"

    step_latency = time.time() - step_start
    state.steps_log.append({
        "step": step_idx,
        "agent": agent_name,
        "role": profile["role"],
        "status": state.sql_execution_status.upper() if agent_name == "SQL_DEVELOPER" else "SUCCESS",
        "latency_ms": round(step_latency * 1000, 1),
    })
    return state


print("✓ Agent pipeline defined")
print(f"  run_agent(state, agent_name) — runs one expert")
print(f"  Pipeline: {' → '.join(MAS_AGENT_ORDER)}")

In [0]:
# ============================================================
# MAS_RAG ORCHESTRATOR — Static 5-Expert Pipeline
# ============================================================
# All 5 professional agents participate on every question.
# Sequential: DOMAIN_EXPERT → DATA_ENGINEER → SQL_DEVELOPER →
#             QUANTITATIVE_ANALYST → QUALITY_AUDITOR
# Each agent sees the question + schema + retrieved context +
# ALL previous agents' outputs. This gives later agents the
# benefit of earlier experts' unique knowledge.
# ============================================================

def run_mas_rag(question_dict: dict) -> dict:
    """
    Execute the MAS_RAG workflow for one question.
    Static pipeline: all 5 experts always participate.

    Args:
        question_dict: dict with id, question, expected_answer, difficulty, etc.

    Returns:
        dict with answer, sql, metrics, retrieved_chunks, agent_outputs, steps, latency, tokens.
    """
    start_time = time.time()

    # Initialize workflow state
    state = WorkflowState(
        question=question_dict["question"],
        question_id=question_dict["id"]
    )

    try:
        # Step 0: Retrieve semantic context (no LLM call)
        state.retrieved_chunks = retrieve_semantic_context(state.question)
        state.retrieved_context_str = format_retrieved_context_for_prompt(state.retrieved_chunks)

        # Run all 5 experts in sequence
        for agent_name in MAS_AGENT_ORDER:
            state = run_agent(state, agent_name)
            if not state.can_answer:
                break  # Early termination if agent says CANNOT_ANSWER

    except Exception as e:
        state.status = "failed"
        state.error = str(e)

    total_latency = time.time() - start_time

    return {
        "answer": state.final_answer,
        "plan": state.agent_outputs.get("DOMAIN_EXPERT", ""),
        "sql_generated": state.sql_generated,
        "sql_results": state.sql_results,
        "sql_execution_status": state.sql_execution_status,
        "retrieved_chunks": state.retrieved_chunks,
        "agent_outputs": state.agent_outputs,
        "steps_log": state.steps_log,
        "prompt_tokens": state.total_prompt_tokens,
        "completion_tokens": state.total_completion_tokens,
        "total_tokens": state.total_prompt_tokens + state.total_completion_tokens,
        "latency_seconds": round(total_latency, 2),
        "llm_calls": state.llm_calls,
        "sql_attempts": state.sql_attempts,
        "agents_used": list(state.agent_outputs.keys()),
        "success": state.status in ("success", "abstained"),
        "status": state.status,
        "error": state.error,
        "is_mock": False,
    }


print("✓ run_mas_rag() defined")
print(f"  Pipeline: {' → '.join(MAS_AGENT_ORDER)}")
print(f"  Static: all 5 agents participate on every question")
print(f"  Retry: SQL_DEVELOPER retries up to 2 attempts on failure")
print(f"  Termination: CANNOT_ANSWER (pre-QA agents) → early abstain; QA uses startswith only")

In [0]:
# ============================================================
# RUN MAS_RAG ARCHITECTURE
# ============================================================

# Respect outer scope (orchestrator sets this); default to all if standalone
if 'QUESTION_FILTER' not in dir():
    QUESTION_FILTER = None
run_questions = [q for q in EVALUATION_QUESTIONS if QUESTION_FILTER is None or q["id"] in QUESTION_FILTER]

print(f"MAS_RAG: {len(run_questions)} questions | 5 expert agents | {MODEL_ENDPOINT}")

all_results = []

for q in run_questions:
    run_id = str(uuid.uuid4())

    try:
        # Execute
        result = run_mas_rag(q)

        # Compute evaluation metrics
        metrics = compute_all_metrics(
            run={"answer": result["answer"], "sql_generated": result.get("sql_generated"),
                 "sql_results": result.get("sql_results"), "success": result["success"],
                 "error": result.get("error")},
            question={"expected_answer": q["expected_answer"], "tables_needed": q.get("tables_needed", []),
                      "answerability_label": q.get("answerability_label", "answerable"),
                      "expected_claims": q.get("expected_claims", []),
                      "question": q["question"]},
            mode="MAS_RAG"
        )

        # Compact output
        sql_st = result.get('sql_execution_status', 'failed')
        print(f"  {q['id']}: {result['latency_seconds']:.1f}s | {result['total_tokens']} tok | SQL:{sql_st} | agents:{result['llm_calls']} calls")

        # Store
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "retrieved", "profile_name": "Multi-Agent Team",
            "model": MODEL_ENDPOINT, "plan": result.get("plan"),
            "sql_generated": result.get("sql_generated"),
            "sql_results": str(result.get("sql_results", ""))[:500],
            "sql_execution_status": result.get("sql_execution_status"),
            "generated_answer": result.get("answer"),
            "retrieved_chunks": get_retrieval_log(result.get('retrieved_chunks', [])),
            "latency_seconds": result["latency_seconds"],
            "prompt_tokens": result.get("prompt_tokens", 0),
            "completion_tokens": result.get("completion_tokens", 0),
            "total_tokens": result["total_tokens"],
            "number_of_model_calls": result["llm_calls"],
            "llm_calls": result["llm_calls"],
            "sql_retry_count": result["sql_attempts"] - 1,
            "sql_attempts": result["sql_attempts"],
            # Thesis KPIs (from question metadata)
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            # Workflow
            "workflow_status": result["status"],
            "success": result["success"], "error": result.get("error"),
            **metrics,
        })

    except Exception as _loop_err:
        print(f"  {q['id']}: ✗ FAILED — {type(_loop_err).__name__}: {_loop_err}")
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "retrieved", "profile_name": "Multi-Agent Team",
            "model": MODEL_ENDPOINT, "plan": None,
            "sql_generated": None, "sql_results": None,
            "sql_execution_status": "failed",
            "generated_answer": None,  # No answer produced — error in "error" field
            "retrieved_chunks": None,
            "latency_seconds": 0.0,
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
            "number_of_model_calls": 0, "llm_calls": 0,
            "sql_retry_count": 0, "sql_attempts": 0,
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            "workflow_status": "failed",
            "success": False, "error": f"{type(_loop_err).__name__}: {_loop_err}",
        })

results_df = pd.DataFrame(all_results)

print(f"\n{'='*70}")
print(f"MAS_RAG RUN COMPLETE: {len(results_df)} runs")
print(f"{'='*70}")
print(f"  SQL success:     {(results_df['sql_execution_status']=='success').sum()}/{len(results_df)}")
print(f"  Avg latency:     {results_df['latency_seconds'].mean():.2f}s")
print(f"  Total tokens:    {results_df['total_tokens'].sum():,}")
print(f"  Avg correctness: {results_df['answer_correctness_proxy'].mean():.2f}")
print(f"  Avg groundedness:{results_df['groundedness_score'].mean():.2f}")

In [0]:
# ============================================================
# CAPTURE MAS_RAG RESULTS + PERSIST TO DELTA
# ============================================================
mas_rag_results_df = None

if "MAS_RAG" in ARCHITECTURES_TO_RUN and 'results_df' in dir() and len(results_df) > 0:
    mas_rag_results_df = results_df.copy()
    all_experiment_results.extend(results_df.to_dict('records'))
    print(f"✓ MAS_RAG captured: {len(mas_rag_results_df)} runs")
else:
    print("⚠ MAS_RAG not run or no results")

_print_cumulative_comparison()

# --- Persist to Delta ---
try:
    if mas_rag_results_df is not None:
        _sdf = spark.createDataFrame(mas_rag_results_df.astype(str))
        _sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(_RESULTS_TABLE)
        print(f"✓ MAS_RAG appended to {_RESULTS_TABLE}")
except Exception as _e:
    print(f"⚠ Delta persist failed: {_e}")

import gc; gc.collect()